# Emulation with Unicorn Engine
### Binary Analysis Beyond Decompilers — Workshop
**Mohamed Jilani Chagra · CyberSphere Congress 2026**

---

## What is Unicorn?

Unicorn is a **CPU emulator** not a full system emulator.  
It emulates **instructions only**. No OS, no syscalls, no file system.

```
Normal execution:   Binary → OS → CPU → Output
                    (you observe from outside)

Unicorn execution:  You map memory → You set registers → CPU runs → You read result
                    (you control everything)
```

**The 7-step Unicorn workflow:**

```python
Step 1:  mu = Uc(UC_ARCH_X86, UC_MODE_64)      # create emulator
Step 2:  mu.mem_map(address, size)              # allocate memory
Step 3:  mu.mem_write(address, bytes)           # write code/data
Step 4:  mu.reg_write(UC_X86_REG_RDI, value)   # set input registers
Step 5:  mu.hook_add(UC_HOOK_CODE, callback)    # watch execution (optional)
Step 6:  mu.emu_start(begin, end)               # run
Step 7:  mu.reg_read(UC_X86_REG_RAX)           # read output
         mu.mem_read(address, size)             # or read memory
```

> **Key insight:** `emu_start(begin, end)` stops when the instruction pointer reaches `end`.  
> You do **not** need a `ret` instruction — just tell it where to stop.

---
## Setup — install and import

In [8]:
# Install if needed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'unicorn', 'capstone', '-q'])
print('Ready!')

Ready!


In [9]:
from unicorn import *
from unicorn.x86_const import *
import capstone
import struct

print(f'unicorn  OK')
print(f'capstone OK')

unicorn  OK
capstone OK


---
## Demo 1 Hello World of Emulation

Two raw x86-64 instructions. No binary file. No OS. Just bytes.

```asm
mov rax, rdi    ; copy input to output  (48 89 F8)
add rax, 5      ; add 5                 (48 83 C0 05)
```

We give it `rdi = 10`, we expect `rax = 15`.

In [15]:
from unicorn import *
from unicorn.x86_const import *

# Raw x86-64 bytes — no binary file needed
# mov rax, rdi   (48 89 F8)
# add rax, 5     (48 83 C0 05)
CODE = b"\x48\x89\xF8"          \
       b"\x48\x83\xC0\x05"

BASE  = 0x1000000    # address where we load the code
STACK = 0x2000000    # address for the stack

# Step 1 — create the emulator: x86 64-bit
mu = Uc(UC_ARCH_X86, UC_MODE_64)

# Step 2 — map memory (must be page-aligned, minimum 0x1000)
mu.mem_map(BASE,  0x10000)   # code region
mu.mem_map(STACK, 0x10000)   # stack region

# Step 3 — write our code bytes into memory
mu.mem_write(BASE, CODE)

# Step 4 — set registers
mu.reg_write(UC_X86_REG_RSP, STACK + 0x8000)  # stack pointer
mu.reg_write(UC_X86_REG_RDI, 10)              # our input = 10

# Step 6 — run from BASE to BASE+len(CODE)
# emu_start stops when IP reaches the end address — no ret needed
mu.emu_start(BASE, BASE + len(CODE))

# Step 7 — read the result
result = mu.reg_read(UC_X86_REG_RAX)
print(f'Input:  {mu.reg_read(UC_X86_REG_RDI)}')
print(f'Result: {result}')    # → 15
print()
print('✓ We never had a binary. We never ran a process.')
print('✓ Just raw bytes in memory, CPU followed them.')

Input:  10
Result: 15

✓ We never had a binary. We never ran a process.
✓ Just raw bytes in memory, CPU followed them.


---
## Demo 2 Hooks: Watching the CPU Work

Same two instructions.  
We add **one callback** now we see every instruction as it executes, with live register values.

This is like a debugger, but written in 5 lines of Python.

In [19]:
from unicorn import *
from unicorn.x86_const import *
import capstone

CODE = b"\x48\x89\xF8"          \
       b"\x48\x83\xC0\x05"

BASE  = 0x1000000
STACK = 0x2000000

mu = Uc(UC_ARCH_X86, UC_MODE_64)
mu.mem_map(BASE,  0x10000)
mu.mem_map(STACK, 0x10000)
mu.mem_write(BASE, CODE)
mu.reg_write(UC_X86_REG_RSP, STACK + 0x8000)
mu.reg_write(UC_X86_REG_RDI, 10)

# Capstone for disassembly inside the hook
cs = capstone.Cs(capstone.CS_ARCH_X86, capstone.CS_MODE_64)

# THE HOOK 
# Called for EVERY instruction before it executes
def hook_code(mu, address, size, user_data):
    # read the bytes at this address and disassemble
    mem = bytes(mu.mem_read(address, size))
    for insn in cs.disasm(mem, address):
        rax = mu.reg_read(UC_X86_REG_RAX)
        rdi = mu.reg_read(UC_X86_REG_RDI)
        print(f'  0x{insn.address:x}: {insn.mnemonic:8} {insn.op_str:20}'
              f' | rax={rax:<5} rdi={rdi}')


# Step 5 — register the hook for ALL instructions
mu.hook_add(UC_HOOK_CODE, hook_code)

mu.emu_start(BASE, BASE + len(CODE))

print(f'\nFinal RAX: {mu.reg_read(UC_X86_REG_RAX)}')
print()
print('✓ One callback. Every instruction. Live register state.')
print('✓ No debugger, no breakpoints pure Python.')

  0x1000000: mov      rax, rdi             | rax=0     rdi=10
  0x1000003: add      rax, 5               | rax=10    rdi=10

Final RAX: 15

✓ One callback. Every instruction. Live register state.
✓ No debugger, no breakpoints pure Python.


---
## Demo 3 Memory Hooks: Track Every Read and Write

Hook types available:

| Hook | Fires when |
|------|------------|
| `UC_HOOK_CODE` | Every instruction executes |
| `UC_HOOK_MEM_WRITE` | Any memory write |
| `UC_HOOK_MEM_READ` | Any memory read |
| `UC_HOOK_BLOCK` | Entry of each basic block |
| `UC_HOOK_INTR` | CPU interrupt |

Now we watch where data goes and where it comes from.

In [ ]:
from unicorn import *
from unicorn.x86_const import *

# Instructions:
#   mov [rsi], rdi    — write rdi into the address in rsi  (48 89 3E)
#   mov rax, [rsi]    — read it back into rax              (48 8B 06)
#   add rax, 0xff     — transform it                       (48 05 FF 00 00 00)
CODE = (b"\x48\x89\x3E"           # mov [rsi], rdi
        b"\x48\x8B\x06"           # mov rax, [rsi]
        b"\x48\x05\xFF\x00\x00\x00")  # add rax, 0xff

BASE  = 0x1000000
STACK = 0x2000000
DATA  = 0x3000000   # a separate region for our data

mu = Uc(UC_ARCH_X86, UC_MODE_64)
mu.mem_map(BASE,  0x10000)
mu.mem_map(STACK, 0x10000)
mu.mem_map(DATA,  0x10000)   # map the data region
mu.mem_write(BASE, CODE)
mu.reg_write(UC_X86_REG_RSP, STACK + 0x8000)
mu.reg_write(UC_X86_REG_RDI, 0x41)   # our value = 'A' = 0x41
mu.reg_write(UC_X86_REG_RSI, DATA)   # rsi = pointer to data region

# MEMORY HOOK 
def hook_mem(mu, access, address, size, value, user_data):
    if access == UC_MEM_WRITE:
        print(f'  [WRITE] 0x{address:x}  size={size}  value=0x{value:x}')
    elif access == UC_MEM_READ:
        data = bytes(mu.mem_read(address, size))
        v = int.from_bytes(data, 'little')
        print(f'  [READ]  0x{address:x}  size={size}  data=0x{v:x}')

mu.hook_add(UC_HOOK_MEM_WRITE, hook_mem)
mu.hook_add(UC_HOOK_MEM_READ,  hook_mem)

mu.emu_start(BASE, BASE + len(CODE))

rax = mu.reg_read(UC_X86_REG_RAX)
print(f'\nFinal RAX: 0x{rax:x}  ({rax})')
print(f'Expected:  0x{0x41 + 0xff:x}  ({0x41 + 0xff})')
print()
print('✓ Every memory access logged. Address, size, value.')
print('✓ This is how analysts track where decryption keys get stored.')

  [WRITE] 0x3000000  size=8  value=0x41
  [READ]  0x3000000  size=8  data=0x41

Final RAX: 0x140  (320)
Expected:  0x140  (320)

✓ Every memory access logged. Address, size, value.
✓ This is how analysts track where decryption keys get stored.


---
## Exercise Decode the Secret

The binary `./secret` contains this function:

```c
void decode(unsigned char *buf, int len, unsigned char key) {
    for (int i = 0; i < len; i++)
        buf[i] ^= key;
}
```

We have an **encoded string** and the **XOR key**.  
Your job: call `decode()` using Unicorn **without running the binary**.

### Linux x86-64 calling convention
```
1st argument → RDI
2nd argument → RSI
3rd argument → RDX
return value → RAX
```

So to call `decode(buf, len, key)`:
```python
RDI = address of buf
RSI = length
RDX = key
```

### Why the fake return address?
When `decode()` finishes, it executes `ret` which pops an address off the stack  
and jumps there. We map a **fake return target** (`RET_ADDR`) and write it on the stack.  
Unicorn's `emu_start` stops when the instruction pointer reaches `RET_ADDR`. ✓

---
### Find the function address
Run this cell to find where `decode()` lives in the binary:

In [23]:
import subprocess

result = subprocess.run(['nm', './secret'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'decode' in line:
        print(line)
        DECODE_ADDR = int(line.split()[0], 16)
        print(f'\ndecode() is at: 0x{DECODE_ADDR:x}')

0000000000401106 T decode

decode() is at: 0x401106


---
### Your turn fill in the TODOs

In [ ]:
from unicorn import *
from unicorn.x86_const import *
import struct

# ── GIVEN ─────────────────────────────────────────────────────
# "CyberSphere" XOR 0x42
ENCODED = bytes([0x01, 0x3b, 0x20, 0x27, 0x30, 0x11,
                 0x32, 0x2a, 0x27, 0x30, 0x27])
KEY = 0x42

# Update this with the address you found above
DECODE_ADDR = 0x401106   # ← from nm output

LOAD_BASE  = 0x400000    # where we load the binary
STACK_BASE = 0x500000    # stack
BUF_ADDR   = 0x600000    # where we put the encoded buffer
RET_ADDR   = 0x700000    # fake return address — decode() will ret here
# ──────────────────────────────────────────────────────────────

with open('./secret', 'rb') as f:
    binary = f.read()

mu = Uc(UC_ARCH_X86, UC_MODE_64)

# TODO 1: map LOAD_BASE with size 0x100000, write binary there

# TODO 2: map STACK_BASE with size 0x100000

# TODO 3: map BUF_ADDR with size 0x1000, write ENCODED there

# TODO 4: map RET_ADDR with size 0x1000
#          (this is just a landing zone — we map it so ret has somewhere to go)

#  5: set up the stack
#   RSP should point to a location where we wrote RET_ADDR as a 64-bit value
mu.mem_write(STACK_BASE + 0x7ff8, struct.pack('<Q', RET_ADDR))
mu.reg_write(UC_X86_REG_RSP, STACK_BASE + 0x7ff8)

# TODO 6: set the three arguments (calling convention: RDI, RSI, RDX)
#   RDI = BUF_ADDR    (pointer to buffer)
#   RSI = len(ENCODED) (length)
#   RDX = KEY         (the XOR key)

# TODO 7: run from DECODE_ADDR, stop at RET_ADDR
#   mu.emu_start(DECODE_ADDR, RET_ADDR)

# TODO 8: read the result from BUF_ADDR
decoded = bytes(mu.mem_read(BUF_ADDR, len(ENCODED)))
print(f'Decoded: {decoded}')
# Expected: b'CyberSphere'

UcError: Invalid memory read (UC_ERR_READ_UNMAPPED)

---
## Complete Solution run only after attempting!

In [29]:
from unicorn import *
from unicorn.x86_const import *
import struct

ENCODED = bytes([0x01, 0x3b, 0x20, 0x27, 0x30, 0x11,
                 0x32, 0x2a, 0x27, 0x30, 0x27])
KEY = 0x42
DECODE_ADDR = 0x401106

LOAD_BASE  = 0x400000
STACK_BASE = 0x500000
BUF_ADDR   = 0x600000
RET_ADDR   = 0x700000

with open('./secret', 'rb') as f:
    binary = f.read()

mu = Uc(UC_ARCH_X86, UC_MODE_64)

# 1. load the binary
mu.mem_map(LOAD_BASE, 0x100000)
mu.mem_write(LOAD_BASE, binary)

# 2. stack
mu.mem_map(STACK_BASE, 0x100000)

# 3. buffer with encoded data
mu.mem_map(BUF_ADDR, 0x1000)
mu.mem_write(BUF_ADDR, ENCODED)

# 4. fake return target
mu.mem_map(RET_ADDR, 0x1000)

# 5. set up stack: write RET_ADDR so ret has somewhere to land
mu.mem_write(STACK_BASE + 0x7ff8, struct.pack('<Q', RET_ADDR))
mu.reg_write(UC_X86_REG_RSP, STACK_BASE + 0x7ff8)

# 6. arguments: decode(buf, len, key)
mu.reg_write(UC_X86_REG_RDI, BUF_ADDR)       # 1st arg: buf pointer
mu.reg_write(UC_X86_REG_RSI, len(ENCODED))   # 2nd arg: length
mu.reg_write(UC_X86_REG_RDX, KEY)            # 3rd arg: XOR key

# 7. run stops when ret jumps to RET_ADDR
mu.emu_start(DECODE_ADDR, RET_ADDR)

# 8. read result from the buffer
decoded = bytes(mu.mem_read(BUF_ADDR, len(ENCODED)))
print(f'Decoded: {decoded.decode()}')

assert decoded == b'CyberSphere', f'Got {decoded} instead'
print('\n✓ We called a function from a binary without running the binary.')
print('✓ This is how malware analysts extract C2 strings, decrypt payloads,')
print('  and understand encoded data — without ever executing the malware.')

Decoded: CyberSphere

✓ We called a function from a binary without running the binary.
✓ This is how malware analysts extract C2 strings, decrypt payloads,
  and understand encoded data — without ever executing the malware.
